# 04b — Model 1: Band-power Logistic Regression (interpretable ground-truth rung)

**This replaces raw logistic regression as the bottom rung.** Raw logistic regression on the raw 3000-sample
epoch was near-chance (balanced accuracy ~0.22 vs 0.20 chance; N3 recall ~0.03) — a linear model on raw
samples cannot represent the *frequency/shape* structure that distinguishes sleep stages, so it neither
learned the task nor gave readable ground truth.

**Why band-power, on Sleep-EDF's own terms.** Sleep stages have well-known **frequency-band signatures**
— most cleanly, **N3 (deep sleep) ↔ delta (0.5–4 Hz)** slow waves. If we hand a linear model those band
powers as features, it becomes a model whose reasoning is **directly readable from physiology**: each
coefficient says "how much does the model rely on the delta band, the spindle band, …". That legibility is
the entire purpose of this rung. It is the supervisor's *start simple and interpretable to understand the
mechanics, then go complex* framing: a model whose ground truth we know, so we can later ask **"do my
attribution methods / CMI recover the band I know the model uses?"** before trusting those methods on the
opaque CNNs.

This rung is **not** meant to be a competitive classifier — it is a **readable ground-truth anchor** for
validating the XAI machinery.

> **Preprocessing consistency — read this.** The band-power feature extraction here is specific to THIS
> baseline rung (its job is band-level interpretability). It does **not** change the ladder's input story:
> the **raw-signal, no-filter decision still governs the CNN and transformer rungs**, which see the raw
> 3000-sample epoch. So the complexity→faithfulness comparison proper runs across the raw-input CNNs and
> transformer; this rung sits alongside as the XAI-mechanism validation.

In [1]:
import sys
from pathlib import Path
# Locate repo root robustly: walk up to the dir containing sleep_edf/ (depth-independent).
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "sleep_edf").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             confusion_matrix, precision_recall_fscore_support)

from sleep_edf.loader import load_sleep_edf              # 20K subsample by default (train)
import sleep_edf.config as cfg
from sleep_edf.bandpower import band_power_features, BAND_NAMES   # 5 named sleep-EEG bands
from sleep_edf.training import run_all_seeds             # shared timed/progress/save driver

CLASS_NAMES = ["W", "N1", "N2", "N3", "REM"]
SEEDS       = [0, 1, 2, 3, 4]
C_L2        = 1.0
MODEL_NAME  = "model1_bandpower"

OUT_DIR = PROJECT_ROOT / "sleep_edf" / "results" / "metrics"
FIG_DIR = PROJECT_ROOT / "sleep_edf" / "results" / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True); FIG_DIR.mkdir(parents=True, exist_ok=True)
np.set_printoptions(precision=3, suppress=True)
print("bands:", BAND_NAMES, "| results ->", OUT_DIR)

bands: ['delta', 'theta', 'alpha', 'sigma', 'beta'] | results -> /Users/luciepasquier/Desktop/IMPERIAL/SUMMER THESIS/Thesis-Repo/sleep_edf/results/metrics


## 1. Band-power feature extraction (5 named bands)

We reduce each 30-s / 3000-sample epoch (Fpz-Cz @100 Hz) to **one number per standard sleep-EEG band**
via a **Welch power-spectral-density (PSD) estimate integrated over each band** (log-power; see
`sleep_edf/bandpower.py`, the single source of truth so the later XAI uses the identical extraction):

| feature | band | Hz |
|---|---|---|
| delta | slow waves | 0.5–4 |
| theta | | 4–8 |
| alpha | | 8–12 |
| sigma | sleep spindles | 12–16 |
| beta | | 16–30 |

**Deliberately tiny and named** — 5 features, not a large opaque vector — so a coefficient (or an
attribution) maps one-to-one onto a physiological band. That legibility is the whole point.

In [2]:
X_train, y_train = load_sleep_edf("train")    # -> logs the 20,000-epoch subsample (seed 42)
X_test,  y_test  = load_sleep_edf("test")     # full 40,145

F_train = band_power_features(X_train)        # (20000, 5) log-band-power
F_test  = band_power_features(X_test)         # (40145, 5)
assert F_train.shape == (20000, 5) and F_test.shape[1] == 5
print("feature matrix:  F_train", F_train.shape, "| F_test", F_test.shape, "(n_epochs x 5 named bands)")

# Reference lines for later.
test_counts = np.bincount(y_test, minlength=cfg.N_CLASSES)
floor = test_counts.max() / len(y_test); chance_balanced = 1.0 / cfg.N_CLASSES
print(f"majority floor (predict {CLASS_NAMES[int(test_counts.argmax())]}): {floor:.4f} "
      f"| balanced-acc chance: {chance_balanced:.3f}")

# Ground-truth PREVIEW (no training): mean band power per stage — is delta elevated in N3?
print(f"\nper-class mean log-band-power (train):")
print(f"{'stage':<6}" + "".join(f"{b:>8}" for b in BAND_NAMES))
for c in range(cfg.N_CLASSES):
    m = F_train[y_train == c].mean(0)
    print(f"{CLASS_NAMES[c]:<6}" + "".join(f"{v:>8.3f}" for v in m))
print("\n=> N3 shows the highest delta and suppressed higher bands — the N3<->delta signature is present.")

[sleep_edf] training load: 20,000-epoch stratified subsample (seed 42)


feature matrix:  F_train (20000, 5) | F_test (40145, 5) (n_epochs x 5 named bands)
majority floor (predict N2): 0.3656 | balanced-acc chance: 0.200

per-class mean log-band-power (train):
stage    delta   theta   alpha   sigma    beta
W        0.319   0.052   0.033   0.018   0.054
N1       0.301   0.065   0.046   0.023   0.043
N2       0.410   0.067   0.031   0.023   0.014
N3       0.480   0.035   0.012   0.007   0.003
REM      0.313   0.081   0.037   0.016   0.031

=> N3 shows the highest delta and suppressed higher bands — the N3<->delta signature is present.


## 2. Ground-truth expectations (stated in ADVANCE)

This is the physiological ground truth the XAI/CMI will later be checked against — recorded **before**
training so the check is honest. Under sleep physiology, which band *should* drive which stage:

- **N3 (deep sleep) ↔ delta (0.5–4 Hz)** — high-amplitude slow waves. **The strongest, least ambiguous
  signature, and the PRIMARY ground-truth validation target:** a trustworthy attribution method, applied
  to this model on N3 epochs, must recover **delta** as the dominant band.
- **N2 ↔ sigma/spindle (12–16 Hz)** (sleep spindles) and **theta** — *secondary expected tendency*.
- **Wake ↔ alpha (8–12 Hz) and beta (16–30 Hz)** — *secondary*.
- **REM ↔ low-amplitude mixed / theta** — weaker, harder to pin to one band (*secondary, loosest*).
- **N1 ↔ theta** — transitional, weak (*secondary, weakest*).

**Primary check: N3 → delta.** The others are secondary expected tendencies, not hard requirements.
After training we read the model's own coefficients (§5) to see which bands it actually weights, and
compare directly to this table — the readable-ground-truth property in action.

## 3. The shared training driver, and this model's `train_one_seed`

Same driver as the raw baseline (`sleep_edf.training.run_all_seeds`): it times seed 0, prints an estimate,
then trains all seeds unattended with a per-seed progress bar and saves each seed as it finishes. Logistic
regression has no epochs, so it ignores the inner-`tick` hook.

The next cell **standardises the 5 band features once** (fit on train only — no leakage; so coefficients
are directly comparable in magnitude across bands) and defines `train_one_seed`. Running it does **not**
train the model.

In [3]:
# Standardise the 5 band features (fit on TRAIN only). Makes coefficients comparable across bands.
scaler = StandardScaler().fit(F_train)
Ftr = scaler.transform(F_train).astype(np.float32)
Fte = scaler.transform(F_test).astype(np.float32)

def train_one_seed(seed, tick=None):
    # 5 band-power features -> 5 sleep stages. max_iter high enough to fully converge (raw needed 1000).
    clf = LogisticRegression(C=C_L2, penalty="l2", solver="lbfgs",
                             max_iter=1000, random_state=seed, n_jobs=-1)
    clf.fit(Ftr, y_train)
    yp = clf.predict(Fte)
    pr, rc, f1c, sup = precision_recall_fscore_support(
        y_test, yp, labels=list(range(cfg.N_CLASSES)), zero_division=0)
    return {
        "accuracy":          float(accuracy_score(y_test, yp)),
        "balanced_accuracy": float(balanced_accuracy_score(y_test, yp)),
        "macro_f1":          float(f1_score(y_test, yp, average="macro")),
        "n_params":          int(clf.coef_.size + clf.intercept_.size),   # 5x5 + 5 = 30
        "converged":         bool(int(clf.n_iter_[0]) < 1000),
        "confusion":         confusion_matrix(y_test, yp, labels=list(range(cfg.N_CLASSES))).tolist(),
        "per_class":         {CLASS_NAMES[c]: {"precision": float(pr[c]), "recall": float(rc[c]),
                                               "f1": float(f1c[c]), "support": int(sup[c])}
                              for c in range(cfg.N_CLASSES)},
        "coef":              clf.coef_.tolist(),        # (5 classes, 5 bands) on standardised log-band-power
        "intercept":         clf.intercept_.tolist(),
    }

print(f"train_one_seed defined; features standardised {Ftr.shape}; params/seed = "
      f"{cfg.N_CLASSES*len(BAND_NAMES)+cfg.N_CLASSES} (ladder low point). Ready to launch (next cell).")

train_one_seed defined; features standardised (20000, 5); params/seed = 30 (ladder low point). Ready to launch (next cell).


## 4. ▶ LAUNCH TRAINING — run this one cell (unattended)

**This is the cell you run to train all 5 seeds.** Times seed 0, estimates, then trains the rest
automatically, saving each seed to `sleep_edf/results/metrics/model1_bandpower_seed{seed}.json`.
`resume=True` skips seeds already on disk — delete those files (or set `resume=False`) to re-train fresh.

In [ ]:
# ▶▶▶ LAUNCH: trains all seeds unattended (times seed 0, estimates, then continues automatically) ◀◀◀
aggregate = run_all_seeds(
    train_one_seed, SEEDS, OUT_DIR, MODEL_NAME,
    summary_keys=["accuracy", "balanced_accuracy", "macro_f1"],
    resume=True,          # delete model1_bandpower_seed*.json (or resume=False) to force a fresh re-train
)

## 5. Did it learn? — and does it use the bands physiology predicts?

Reads the saved results (works after the launch cell / a kernel restart). We check accuracy vs the ~0.37
floor and balanced accuracy vs 0.20 chance, per-class precision/recall/F1 with **N1 and N3 explicit**, and
— the point of this rung — the **learned coefficients per class × band**, compared to the §2 expectations.
Expectation: this model **actually learns** (balanced acc well above chance, N3 recall far above the raw
model's ~0.03).

In [ ]:
import json
agg = json.load(open(OUT_DIR / f"{MODEL_NAME}_aggregate.json"))
per_seed = agg["per_seed"]; s = agg["summary"]
print(f"parameter count (ladder low point): {per_seed[str(SEEDS[0])]['n_params']}")
print(f"accuracy      : {s['accuracy']['mean']:.4f} ± {s['accuracy']['std']:.4f}   "
      f"(floor {floor:.4f}; margin {s['accuracy']['mean']-floor:+.4f})")
print(f"balanced acc  : {s['balanced_accuracy']['mean']:.4f} ± {s['balanced_accuracy']['std']:.4f}   "
      f"(chance {chance_balanced:.3f}; margin {s['balanced_accuracy']['mean']-chance_balanced:+.4f})")
print(f"macro-F1      : {s['macro_f1']['mean']:.4f} ± {s['macro_f1']['std']:.4f}")
print(f"converged all seeds: {all(per_seed[str(sd)]['converged'] for sd in SEEDS)}")

print(f"\n{'stage':<6}{'precision':>11}{'recall':>9}{'f1':>9}{'support':>10}")
print('-'*45)
for cn in CLASS_NAMES:
    P = np.mean([per_seed[str(sd)]['per_class'][cn]['precision'] for sd in SEEDS])
    R = np.mean([per_seed[str(sd)]['per_class'][cn]['recall']    for sd in SEEDS])
    F = np.mean([per_seed[str(sd)]['per_class'][cn]['f1']        for sd in SEEDS])
    print(f"{cn:<6}{P:>11.3f}{R:>9.3f}{F:>9.3f}{per_seed[str(SEEDS[0])]['per_class'][cn]['support']:>10}")
print("\nCheck N3 recall (raw model gave ~0.03) and balanced acc vs 0.20 chance.")

In [ ]:
cm = np.sum([np.array(per_seed[str(sd)]['confusion']) for sd in SEEDS], axis=0)
cmn = cm / cm.sum(axis=1, keepdims=True)
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
for a, M, title, fmt in [(ax[0], cm, "Confusion (counts, summed over seeds)", "d"),
                         (ax[1], cmn, "Row-normalised (recall per true stage)", ".2f")]:
    a.imshow(M, cmap="Blues", vmin=0, vmax=(M.max() if fmt=="d" else 1))
    for i in range(5):
        for j in range(5):
            a.text(j, i, format(M[i, j], fmt), ha="center", va="center", fontsize=8,
                   color="white" if M[i, j] > (M.max()*0.5 if fmt=="d" else 0.5) else "black")
    a.set_xticks(range(5)); a.set_xticklabels(CLASS_NAMES); a.set_yticks(range(5)); a.set_yticklabels(CLASS_NAMES)
    a.set_xlabel("predicted"); a.set_ylabel("true"); a.set_title(title)
fig.tight_layout(); fig.savefig(FIG_DIR / "sleep_edf_04b_bandpower_confusion.png", dpi=150, bbox_inches="tight")
plt.show(); print("saved:", (FIG_DIR / "sleep_edf_04b_bandpower_confusion.png").relative_to(PROJECT_ROOT))

In [ ]:
# Learned coefficients (class x band), averaged across seeds, on standardised log-band-power.
# coef[class, band] > 0  =>  above-average power in that band pushes the prediction TOWARD that stage.
coef = np.mean([np.array(per_seed[str(sd)]['coef']) for sd in SEEDS], axis=0)   # (5 classes, 5 bands)

fig, ax = plt.subplots(figsize=(6.5, 4.5))
vmax = np.abs(coef).max()
im = ax.imshow(coef, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
for i in range(coef.shape[0]):
    for j in range(coef.shape[1]):
        ax.text(j, i, f"{coef[i,j]:+.2f}", ha="center", va="center", fontsize=8,
                color="white" if abs(coef[i,j])>vmax*0.6 else "black")
ax.set_xticks(range(len(BAND_NAMES))); ax.set_xticklabels(BAND_NAMES)
ax.set_yticks(range(len(CLASS_NAMES))); ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel("band feature"); ax.set_ylabel("stage"); ax.set_title("Logistic-regression coefficients (class x band)")
fig.colorbar(im, ax=ax, fraction=0.046, label="weight (+ = pushes toward stage)")
fig.tight_layout(); fig.savefig(FIG_DIR / "sleep_edf_04b_bandpower_coeffs.png", dpi=150, bbox_inches="tight")
plt.show()

EXPECT = {"W":"alpha/beta", "N1":"theta", "N2":"sigma/theta", "N3":"delta", "REM":"theta/mixed"}
print("top positively-weighted band per stage  vs  physiological expectation:")
for c, cn in enumerate(CLASS_NAMES):
    top = BAND_NAMES[int(coef[c].argmax())]
    print(f"  {cn:<4} model-top: {top:<6}   expected: {EXPECT[cn]}")
print("\nPRIMARY check: N3's top band should be DELTA (the clean ground-truth target for the XAI).")

## 6. Verdict

Fill in from the numbers above once trained. Two things this rung must show:
1. **It functions** — balanced accuracy meaningfully above 0.20 chance and N3 recall far above the raw
   model's ~0.03, unlike raw logistic regression which was near-chance.
2. **It is readable ground truth** — the coefficients weight the physiologically-expected bands, above all
   **N3 → delta** (the primary target the attribution methods will later be validated against).

If both hold, this is a legitimate simple baseline AND the interpretable anchor for checking the XAI/CMI
machinery before it is trusted on the opaque CNNs. Seed-to-seed variance is expected ~0 (convex model).